## Generate System Prompt with available tools

In [1]:
import pandas as pd
import json
import random

# File paths
SYSTEM_PROMPT_PATH = "D:\\Uni Study\\Sem-6\\6.My Code\\1.Single_turn\\system.md"
EXCEL_PATH = "D:\\Uni Study\\Sem-6\\6.My Code\\1.Single_turn\\2(1).Single-turn-new.xlsx"
JSON_PATH = "D:\\Uni Study\\Sem-6\\7.My Datasets\\Func Schema\\3.functions_by_scenario_v3.json"
OUTPUT_EXCEL_PATH = "D:\\Uni Study\\Sem-6\\6.My Code\\1.Single_turn\\system_prompts_output.xlsx"

# Read system prompt template
with open(SYSTEM_PROMPT_PATH, 'r') as f:
    system_prompt_template = f.read()

# Read JSON file with tools
with open(JSON_PATH, 'r') as f:
    tools_data = json.load(f)

# Read Excel file
df = pd.read_excel(EXCEL_PATH)

# Function to extract tools based on tool_name or select random category
def get_tools_for_query(tool_name):
    if pd.isna(tool_name):  # If no tool_name, pick a random category
        category = random.choice(list(tools_data.keys()))
        tools = tools_data[category][:3]  # Select first 3 tools from the category
    else:
        # Strip whitespace from tool_name
        tool_name = tool_name.strip()
        
        # Find the category containing the tool_name
        found_category = None
        target_tool = None
        
        # Search through all categories and tools
        for category, tool_list in tools_data.items():
            for tool in tool_list:
                # Check if this tool matches our target tool_name
                if (tool.get('function', {}).get('name') == tool_name):
                    found_category = category
                    target_tool = tool
                    break
            if found_category:  # Exit outer loop if found
                break
        
        if found_category:
            # Get all tools from the found category
            category_tools = tools_data[found_category]
            
            # Start with the target tool first, then add others
            selected_tools = [target_tool]
            
            # Add the remaining tools from the same category (excluding the target tool)
            for tool in category_tools:
                if tool.get('function', {}).get('name') != tool_name:
                    selected_tools.append(tool)
                    if len(selected_tools) >= 3:  # Limit to 3 total tools
                        break
            
            tools = selected_tools
        else:
            # If tool_name not found, pick a random category
            print(f"Tool '{tool_name}' not found, selecting random category")
            category = random.choice(list(tools_data.keys()))
            tools = tools_data[category][:3]
    
    # Keep tools as they are in the JSON file
    return json.dumps(tools)

# Generate system prompts with only the tools
system_prompts = []
tools_used = []  # Track which tools were included

for _, row in df.iterrows():
    tool_name = row['tool_name']
    
    # Get tools for the query
    tools_json = get_tools_for_query(tool_name)
    
    # Parse the JSON to extract tool names for tracking
    tools_list = json.loads(tools_json)
    tool_names = [tool.get('function', {}).get('name', 'unknown') for tool in tools_list]
    tools_used.append(', '.join(tool_names))
    
    # Replace {tools} in the system prompt template
    system_prompt = system_prompt_template.replace("{tools}", tools_json)
    
    system_prompts.append(system_prompt)

# Create new DataFrame with id, Query, System_Prompt and tools
output_df = pd.DataFrame({
    'id': df['id'],
    'Query': df['Query'],
    'System_Prompt': system_prompts,
    'Tools_Used': tools_used
})

# Save to new Excel file
output_df.to_excel(OUTPUT_EXCEL_PATH, index=False)

print(f"System prompts generated and saved to {OUTPUT_EXCEL_PATH}")

Tool 'process_leave_request' not found, selecting random category
Tool 'conduct_recruitment_process' not found, selecting random category
Tool 'manage_customer_campaign' not found, selecting random category
Tool 'generate_customer_segmentation_report' not found, selecting random category
Tool 'process_purchase_order' not found, selecting random category
Tool 'conduct_inventory_audit' not found, selecting random category
Tool 'generate_financial_statements' not found, selecting random category
Tool 'forecast_revenue' not found, selecting random category
Tool 'manage_user_accounts' not found, selecting random category
Tool 'deploy_software' not found, selecting random category
Tool 'manage_supplier_contracts' not found, selecting random category
Tool 'evaluate_vendor_performance' not found, selecting random category
Tool 'conduct_spend_analysis' not found, selecting random category
Tool 'automate_lead_assignment' not found, selecting random category
Tool 'generate_contract_renewal_remind

## Conversation - role (system,user,assistant & content)

In [2]:
import pandas as pd
import json

# File paths
EXCEL_PATH = "D:\\Uni Study\\Sem-6\\6.My Code\\1.Single_turn\\2(1).Single-turn-new.xlsx"
OUTPUT_EXCEL_PATH = "D:\\Uni Study\\Sem-6\\6.My Code\\1.Single_turn\\new_conversation_output.xlsx"

# Read Excel file
df = pd.read_excel(EXCEL_PATH)

# Function to format conversation
def create_conversation(row):
    system_prompt = row['System_Prompt']
    query = row['Query']
    ground_truth = row['Ground_truth']
    tool_needed = row['tool_needed']
    
    # Format assistant response based on tool_needed
    if tool_needed:
        try:
            # Parse Ground_truth as JSON
            parsed_ground_truth = json.loads(ground_truth)
            # Check if parsed_ground_truth is a list and not empty
            if isinstance(parsed_ground_truth, list) and len(parsed_ground_truth) > 0:
                tool_call = parsed_ground_truth[0]  # Get the first tool call
                assistant_content = f"""<tool_call>
{json.dumps(tool_call, indent=2)}
</tool_call>"""
            else:
                # Fallback to plain text if not a valid list
                assistant_content = ground_truth
        except (json.JSONDecodeError, TypeError):
            # Fallback to plain text if JSON parsing fails or ground_truth is not a string
            assistant_content = ground_truth
    else:
        assistant_content = ground_truth  # Keep as plain text
    
    # Create conversation list of dictionaries
    conversation = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": query},
        {"role": "assistant", "content": assistant_content}
    ]
    
    # Convert to JSON string for Excel storage
    return json.dumps(conversation, indent=2, ensure_ascii=False)

# Generate conversations
df['Conversation'] = df.apply(create_conversation, axis=1)

# Save to new Excel file
df.to_excel(OUTPUT_EXCEL_PATH, index=False)

print(f"Conversations generated and saved to {OUTPUT_EXCEL_PATH}")

Conversations generated and saved to D:\Uni Study\Sem-6\6.My Code\1.Single_turn\new_conversation_output.xlsx


## Split & Dataset push to HF
### (Define what columns to push)

In [ ]:
######## No Splitting
import pandas as pd
from datasets import Dataset, DatasetDict
from huggingface_hub import login

# Step 1: Login to Hugging Face Hub
login(token="")  # Replace with your actual token

# Step 2: Load your data (from Excel, CSV, or DataFrame)
df = pd.read_excel("D:\\Uni Study\\Sem-6\\6.My Code\\1.Single_turn\\2(1).Single-turn-new.xlsx")  # or pd.read_csv()

# Step 3: Select relevant columns (optional)
relevant_columns = ['id', 'Turn Type', 'Prompt_type', 'tool_needed', 'tool_call', 'tool_name', 'Query', 'Ground_truth', 'Missing_para', 'tools_in_system_prompt', 'System_Prompt', 'Conversation']
df_filtered = df[relevant_columns]

# Step 4: Convert to Hugging Face Dataset
dataset = Dataset.from_pandas(df_filtered)

# Step 5: Push to Hub
dataset.push_to_hub(
    "kunjanshah/single_turn_function_calling",
    private=False  # Set to True if you want a private dataset
)

print("Dataset successfully pushed to Hugging Face Hub!")

c:\Users\KUNJAN SHAH\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Creating parquet from Arrow format: 100%|██████████| 2/2 [00:00<00:00, 45.82ba/s]
Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

Processing Files (0 / 1)                :  26%|██▌       |  584kB / 2.27MB,  583kB/s  

Processing Files (0 / 1)                :  77%|███████▋  | 1.75MB / 2.27MB, 1.25MB/s  

Processing Files (1 / 1)                : 100%|██████████| 2.27MB / 2.27MB, 1.27MB/s  



Processing Files (1 / 1)                : 100%|██████████| 2.27MB / 2.27MB,  947kB/s  
New Data Upload                         : 100%|██████████| 2.27MB / 2.27MB,  947kB/s  
                                        : 100%|██████████| 2.27MB / 2.27MB            
Uploading the dataset s

Dataset successfully pushed to Hugging Face Hub!


In [ ]:
######## With Splitting
import pandas as pd
from datasets import Dataset
from huggingface_hub import login
import numpy as np
import sys

# File paths
EXCEL_PATH = "D:\\Uni Study\\Sem-6\\6.My Code\\1.Single_turn\\2(1).Single-turn-new.xlsx"

# Step 1: Read the Excel file
df = pd.read_excel(EXCEL_PATH)
total_rows = len(df)
print(f"Total rows in dataset: {total_rows}")

# Step 2: Shuffle and split the dataset
df_shuffled = df.sample(frac=1, random_state=42).reset_index(drop=True)

# Split into train, validation, and test sets (80% train, 10% validation, 10% test)
train_frac = 0.8
valid_frac = 0.1
test_frac = 0.1

# Calculate exact split sizes to ensure they sum to total rows
train_size = int(train_frac * total_rows)
valid_size = int(valid_frac * total_rows)
# Ensure test gets remaining rows so the total is exact
test_size = total_rows - train_size - valid_size

# Create train, validation, and test dataframes
df_train = df_shuffled[:train_size]
df_valid = df_shuffled[train_size:train_size + valid_size]
df_test = df_shuffled[train_size + valid_size:]

# Validation check
actual_total = len(df_train) + len(df_valid) + len(df_test)
print(f"Validation: Expected {total_rows} rows, got {actual_total} rows")

# Calculate memory usage
def get_size_in_mb(df):
    return df.memory_usage(deep=True).sum() / (1024 * 1024)

train_mb = get_size_in_mb(df_train)
valid_mb = get_size_in_mb(df_valid)
test_mb = get_size_in_mb(df_test)
total_mb = train_mb + valid_mb + test_mb

# Print detailed split information
print(f"Training set: {len(df_train)} rows ({len(df_train)/total_rows:.1%}), {train_mb:.2f} MB")
print(f"Validation set: {len(df_valid)} rows ({len(df_valid)/total_rows:.1%}), {valid_mb:.2f} MB")
print(f"Test set: {len(df_test)} rows ({len(df_test)/total_rows:.1%}), {test_mb:.2f} MB")
print(f"Total dataset size: {total_mb:.2f} MB")

# Step 3: Convert each DataFrame to Hugging Face Dataset
relevant_columns = ['id', 'Turn Type', 'Prompt_type', 'tool_needed', 'tool_call', 'tool_name', 
                     'Query', 'Ground_truth', 'Missing_para', 'tools_in_system_prompt', 'System_Prompt', 'Conversation']

train_dataset = Dataset.from_pandas(df_train[relevant_columns])
valid_dataset = Dataset.from_pandas(df_valid[relevant_columns])
test_dataset = Dataset.from_pandas(df_test[relevant_columns])

# Step 4: Create a DatasetDict for the split datasets
from datasets import DatasetDict

dataset_dict = DatasetDict({
    'train': train_dataset,
    'validation': valid_dataset,
    'test': test_dataset
})

# Print dataset information
print("\nDataset splits information:")
print(f"Train: {len(train_dataset)} examples")
print(f"Validation: {len(valid_dataset)} examples")
print(f"Test: {len(test_dataset)} examples")

# Step 5: Log in to Hugging Face Hub
login(token="")  # Use your actual token with write access

# Step 6: Create dataset card with metadata and statistics
dataset_description = f"""
# Single Turn Function Calling Dataset

## Dataset Description
This dataset contains {total_rows} examples of single-turn conversations for function calling with tool use.

## Dataset Statistics
- **Total examples**: {total_rows}
- **Train split**: {len(train_dataset)} examples ({len(df_train)/total_rows:.1%})
- **Validation split**: {len(valid_dataset)} examples ({len(df_valid)/total_rows:.1%})
- **Test split**: {len(test_dataset)} examples ({len(df_test)/total_rows:.1%})

## Dataset Size
- **Total size**: {total_mb:.2f} MB
- **Training set**: {train_mb:.2f} MB
- **Validation set**: {valid_mb:.2f} MB
- **Test set**: {test_mb:.2f} MB

## Dataset Features
- `id`: Unique identifier for each example
- `Turn Type`: Type of conversation turn
- `Prompt_type`: Type of prompt used
- `tool_needed`: Whether a tool is needed (boolean)
- `tool_call`: Tool call information
- `tool_name`: Name of the tool used
- `Query`: User query
- `Ground_truth`: Expected response
- `Missing_para`: Any missing parameters
- `tools_in_system_prompt`: Tools available in the system prompt
- `System_Prompt`: System prompt used
- `Conversation`: Full conversation including system, user, and assistant messages

## Tool Distribution
{df['tool_name'].value_counts().head(10).to_markdown()}

## License
This dataset is provided for research purposes.
"""

# Step 7: Push dataset to Hugging Face Hub with the split configuration and metadata
dataset_dict.push_to_hub(
    "kunjanshah/new_single_turn_function_calling_split", 
    private=False
)
print("\nSplit dataset successfully pushed to Hugging Face Hub")

c:\Users\KUNJAN SHAH\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Total rows in dataset: 1418
Validation: Expected 1418 rows, got 1418 rows
Training set: 1134 rows (80.0%), 14.50 MB
Validation set: 141 rows (9.9%), 1.76 MB
Test set: 143 rows (10.1%), 1.86 MB
Total dataset size: 18.12 MB

Dataset splits information:
Train: 1134 examples
Validation: 141 examples
Test: 143 examples


Creating parquet from Arrow format: 100%|██████████| 2/2 [00:00<00:00, 71.94ba/s]
Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

Processing Files (0 / 1)                :  19%|█▉        |  382kB / 1.98MB,   ???B/s  


Processing Files (0 / 1)                :  48%|████▊     |  959kB / 1.98MB,  965kB/s  
Processing Files (0 / 1)                :  78%|███████▊  | 1.54MB / 1.98MB, 1.45MB/s  

Processing Files (1 / 1)                : 100%|██████████| 1.98MB / 1.98MB, 1.34MB/s  



Processing Files (1 / 1)                : 100%|██████████| 1.98MB / 1.98MB,  891kB/s  
New Data Upload                         : 100%|██████████| 1.60MB / 1.60MB,  891kB/s  
                                        : 100%|██████████| 1.98MB / 1.98MB            
Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 180.70ba/s]
Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

Processing Files (1 / 1)                : 100%|█████


Split dataset successfully pushed to Hugging Face Hub
